In [ ]:
from openai import OpenAI
import time
import statistics

client = OpenAI("Insert your API key here")

Here is the single agent submodule that evaluates the full text of a paper based on the provided evaluation guidelines and scoring criteria. It returns an average score

In [ ]:
def single_agent_submodule(full_text : str, evaluation_guidelines: str, scoring_criteria: str) -> int:
    """
    This agent evaluates the full_text based on the provided evaluation_guidelines and scoring_criteria.
    It returns a score between 1 and 10, where 1 is the worst and 10 is the best.
    """
    prompt = f"""
You are an expert reviewer for a top-tier artificial intelligence and machine learning conference. Read the following paper carefully and assign one overall score based solely on the criteria listed below.

Scoring Scale (return only one of these values):
{scoring_criteria}

Evaluation Criteria:
{evaluation_guidelines}

Consider all criteria holistically. Use the full scoring scale—including low scores where appropriate—and do not hesitate to assign either low or high scores when justified. Neither penalize nor reward unduly.

Output format: Return only a single integer from the scoring scale above. Do not include any explanation, commentary, or justification.

Paper to review:
{full_text}
    """
    
    ratings = []
    for i in range(5):
        while True:
            try:
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": prompt}],
                    model="your_model_name_here",
                    temperature=1,
                )

                break  # esce se la chiamata va a buon fine
            except Exception as e:
                print(f"Error during API call: {e}. Retrying in 10 seconds...")
                time.sleep(10)

        score = int(chat_completion.choices[0].message.content)
        ratings.append(score)

    
    return statistics.mean(ratings)

This is the multi-agent version of the submodule, where we first generate separate reviews for different aspects of the paper and then use those reviews to assign a final score.
You can easily add or remove aspects to review by modifying the prompts and the final prompt accordingly.

In [ ]:

def multi_agent_submodule(full_text : str, evaluation_guidelines: str, scoring_criteria: str) -> int:
    """
    This agent evaluates the full_text based on the provided evaluation_guidelines and scoring_criteria.
    It returns a score between 1 and 10, where 1 is the worst and 10 is the best.
    """
    prompt = """

You are an expert reviewer for a top-tier artificial intelligence and machine learning conference.
You are given a set of partial reviews, each one evaluating different aspects of the paper (e.g., originality, motivation, technical quality, empirical validation, clarity).
Read these reviews carefully and assign one overall score based solely on the criteria listed below.

Scoring Scale (return only one of these values):
{scoring_criteria}

Evaluation Criteria:
{evaluation_guidelines}

Consider all criteria holistically. The reviews may highlight strengths, weaknesses, or points of contention, but you must independently weigh the evidence in the paper when assigning the final score. Use the full scoring scale—including low scores where appropriate—and do not hesitate to assign either low or high scores when justified. Neither penalize nor reward unduly.

Output format: Return only a single integer from the scoring scale above. Do not include any explanation, commentary, or justification.

Paper to review:
{full_text}

Reviews:
{clarity_review}
{originality_review}
{technical_review}
{motivation_review}
{empirical_review}
"""

    ratings = []
    reviews = []
    for i in range(5):
        clarity_prompt = f"""
You are a reviewer for the ICLR conference. Your task is to evaluate the clarity, writing quality, and logical structure of the following paper.

Read the paper carefully and provide a critical analysis addressing:
- Is the main contribution clearly stated?
- Is the writing precise, well-formulated, and accessible?
- Does the text follow a coherent and logical structure?

Base your analysis solely on the content of the paper, without assuming information not provided.

Input paper:
{full_text}
"""
        while True:
            try:
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": clarity_prompt}],
                    model="gpt-4o-mini",
                )
                break
            except Exception as e:
                print(f"Error during clarity API call: {e}. Retrying in 10 seconds...")
                time.sleep(10)

        clarity_review = chat_completion.choices[0].message.content

        originality_prompt = f"""
    You are an ICLR reviewer. Your role is to assess the originality and potential impact of the research described in the following paper.

    Please address the following:
    - Does the work present novel ideas, methods, or perspectives?
    - Is the problem addressed meaningful and relevant?
    - Does the paper suggest that the work could significantly contribute to its field?

    Base your analysis solely on the content of the paper, without assuming information not provided.

    Input Paper:
    {full_text}
    """
        while True:
            try:
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": originality_prompt}],
                    model="gpt-4o-mini",
                )
                break
            except Exception as e:
                print(f"Error during originality API call: {e}. Retrying in 10 seconds...")
                time.sleep(10)
        originality_review = chat_completion.choices[0].message.content

        technical_prompt = f"""
    You are reviewing a paper for the ICLR conference. Your role is to assess the technical soundness of the work as described in the paper.

    Analyze the following:
    - Do the claims and methods appear well-founded and reasonable?
    - Are there any logical inconsistencies or unsupported assumptions?
    - Does the approach seem methodologically appropriate?

    Base your analysis solely on the content of the paper, without assuming information not provided.

    Input Paper:
    {full_text}
    """
        while True:
            try:
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": technical_prompt}],
                    model="gpt-4o-mini",
                )
                break
            except Exception as e:
                print(f"Error during technical API call: {e}. Retrying in 10 seconds...")
                time.sleep(10)

        technical_review = chat_completion.choices[0].message.content

        motivation_prompt = f"""
    You are a reviewer for the ICLR conference. Your task is to evaluate the motivation and problem framing of the following paper.

    Read the paper carefully and provide a critical analysis addressing:
    - Is the research problem clearly motivated and well justified?
    - Is the problem grounded in relevant prior work or context?
    - Does the paper make it clear why this problem is important?

    Base your analysis solely on the content of the paper, without assuming information not provided.

    Input paper:
    {full_text}
    """

        while True:
            try:
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": motivation_prompt}],
                    model="gpt-4o-mini",
                )
                break
            except Exception as e:
                print(f"Error during motivation API call: {e}. Retrying in 10 seconds...")
                time.sleep(10)

        motivation_review = chat_completion.choices[0].message.content

        empirical_prompt = f"""
    You are a reviewer for the ICLR conference. Your task is to evaluate the empirical validation of the following paper.

    Read the paper carefully and provide a critical analysis addressing:
    - Are the experiments, results, or evaluations described clearly and rigorously?
    - Do the reported results support the paper’s main claims?
    - Is the empirical evidence sufficient to demonstrate the effectiveness of the approach?

    Base your analysis solely on the content of the paper, without assuming information not provided.

    Input paper:
    {full_text}
    """
        while True:
            try:
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": empirical_prompt}],
                    model="gpt-4o-mini",
                )
                break
            except Exception as e:
                print(f"Error during empirical validation API call: {e}. Retrying in 10 seconds...")
                time.sleep(10)

        empirical_review = chat_completion.choices[0].message.content
        
        final_prompt = prompt.format(
            scoring_criteria=scoring_criteria,
            evaluation_guidelines=evaluation_guidelines,
            full_text=full_text,
            clarity_review=clarity_review,
            originality_review=originality_review,
            technical_review=technical_review,
            motivation_review=motivation_review,
            empirical_review=empirical_review
        )
        
        while True:
            try:
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": final_prompt}],
                    model="gpt-4o-mini",
                )
                score = int(chat_completion.choices[0].message.content)
                break
            except Exception as e:
                print(f"Error during final scoring API call: {e}. Retrying in 10 seconds...")
                time.sleep(10)
        ratings.append(score)
        reviews.append({
            "clarity": clarity_review,
            "originality": originality_review,
            "technical": technical_review,
            "motivation": motivation_review,
            "empirical": empirical_review
        })
        
    return {"score": statistics.mean(ratings), "reviews": reviews}




In [ ]:

if __name__ == "__main__":
    
    guidelines = """Insert your evaluation guidelines here."""
    criteria = """Insert your scoring criteria here."""
    full_text = """Insert the full text of the paper to review here."""
    
    score_single = single_agent_submodule(full_text, guidelines, criteria)
    multi_agent_result = multi_agent_submodule(full_text, guidelines, criteria)
    
    final_score = int((score_single + multi_agent_result["score"]) / 2)
    
    print(f"Final Score for the paper: {final_score}")
    print("Multi-agent reviews:")
    for idx, review in enumerate(multi_agent_result["reviews"]):
        print(f"Review {idx+1}:")
        print(f"Clarity: {review['clarity']}")
        print(f"Originality: {review['originality']}")
        print(f"Technical Quality: {review['technical']}")
        print(f"Motivation: {review['motivation']}")
        print(f"Empirical Validation: {review['empirical']}")
        print("-" * 40)
    